# Can we predict next-month arrears?
Target: `arrears_next_month`. Follows `checklist.md`.

**Feature timing (checklist 7).** `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (and it is almost a proxy for the target). It is dropped, as is the identifier `meter_ref`. `billing_month` is used only for splitting. Allowed features: `tariff, occupants, kwh, bill_eur, late_payments_12m`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv',parse_dates=['billing_month']).sort_values('billing_month').reset_index(drop=True)
TARGET='arrears_next_month'
NUM=['occupants','kwh','bill_eur','late_payments_12m']; CAT=['tariff']; FEATS=NUM+CAT
print(df.shape, df[TARGET].mean())
print(df.groupby('final_notice_sent')[TARGET].mean())  # why it is excluded

## 1. Test set (checklist 1)
Model is meant for future months and accounts appear in many rows, so the test set is the **latest months, for accounts held out entirely**: 20% of accounts are randomly assigned to test; test rows are those accounts' rows in the last 4 billing months. Held-out accounts' earlier rows are discarded, and training uses only non-test accounts' rows before the test period. So training sees neither test accounts nor test-period months.

In [ ]:
rng=np.random.RandomState(SEED)
meters=df.meter_ref.unique()
test_meters=set(rng.choice(meters,size=int(0.2*len(meters)),replace=False))
months=np.sort(df.billing_month.unique())
cutoff=months[-4]
is_test_m=df.meter_ref.isin(test_meters)
test=df[is_test_m & (df.billing_month>=cutoff)].copy()
train=df[~is_test_m & (df.billing_month<cutoff)].copy().reset_index(drop=True)
assert not set(train.meter_ref)&set(test.meter_ref)
assert train.billing_month.max()<test.billing_month.min()
print(len(train),len(test),train[TARGET].mean(),test[TARGET].mean())
# test is NOT looked at again until step 5

## 2. Cross-validation inside training rows (checklist 2)
Same structure: 5 expanding-window time folds over training months; in each fold the validation block is a later set of months, and only accounts *not* in the validation block's account subset are used for fitting (training rows are earlier months of other accounts; validation rows are later months of a held-out account group). Preprocessing lives inside the sklearn `Pipeline`, so it is fit within each fold (checklist 3).

In [ ]:
tm=np.sort(train.billing_month.unique())
blocks=np.array_split(tm[len(tm)//2:],5)   # validation month blocks
mgroups=pd.Series(train.meter_ref.unique()).sample(frac=1,random_state=SEED).reset_index(drop=True)
fold_of=dict(zip(mgroups,np.arange(len(mgroups))%5))
train['grp']=train.meter_ref.map(fold_of)
def folds():
    for k,b in enumerate(blocks):
        va=train.index[(train.billing_month.isin(b))&(train.grp==k)]
        tr=train.index[(train.billing_month<b.min())&(train.grp!=k)]
        yield tr,va
for tr,va in folds():
    assert not set(train.meter_ref[tr])&set(train.meter_ref[va])
    assert train.billing_month[tr].max()<train.billing_month[va].min()
    print(len(tr),len(va),train[TARGET][va].mean().round(3))

## 3. Candidate models (checklist 4) — fixed before any scoring
1. `baseline`: predicts the training prevalence (trivial).
2. `logit_C0.1`, `logit_C1`, `logit_C10`: logistic regression, one-hot tariff + standardised numerics.
3. `hgb_shallow`, `hgb_deep`: gradient boosting (max_depth 3 / 6, lr 0.05).

Primary selection metric: CV average precision (PR-AUC; arrears is the rare class). Threshold: the one maximising F1 on pooled out-of-fold predictions of the chosen model. Features: all five allowed features (no feature search).

In [ ]:
pre=lambda: ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
CANDS={
 'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logit_C0.1':Pipeline([('p',pre()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logit_C1':Pipeline([('p',pre()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'logit_C10':Pipeline([('p',pre()),('m',LogisticRegression(C=10,max_iter=1000))]),
 'hgb_shallow':Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,random_state=SEED))]),
 'hgb_deep':Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,random_state=SEED))]),
}
oof={}; rows=[]
for name,mdl in CANDS.items():
    p=pd.Series(np.nan,index=train.index); ap=[];auc=[]
    for tr,va in folds():
        m=clone(mdl).fit(train.loc[tr,FEATS],train.loc[tr,TARGET])
        p[va]=m.predict_proba(train.loc[va,FEATS])[:,1]
        ap.append(average_precision_score(train.loc[va,TARGET],p[va])); auc.append(roc_auc_score(train.loc[va,TARGET],p[va]))
    oof[name]=p
    rows.append((name,np.mean(ap),np.std(ap),np.mean(auc)))
cv=pd.DataFrame(rows,columns=['model','cv_AP','AP_sd','cv_AUC']).set_index('model'); cv

## 4. Choice (checklist 5) — from CV only

In [ ]:
best=cv.cv_AP.idxmax()
mask=oof[best].notna(); y=train.loc[mask,TARGET]; p=oof[best][mask]
ths=np.linspace(0.02,0.9,89)
f1s=[f1_score(y,p>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| CV AP',round(cv.cv_AP[best],3),'vs baseline',round(cv.cv_AP['baseline'],3))
print('chosen threshold:',round(thr,2),'CV F1',round(max(f1s),3))

**Decision (recorded before test scoring):** the model with highest CV average precision (printed above), all five features, with the CV-F1-maximising threshold printed above. These are now frozen.

## 5. Final fit and single test evaluation (checklist 6)

In [ ]:
final=clone(CANDS[best]).fit(train[FEATS],train[TARGET])
pt=final.predict_proba(test[FEATS])[:,1]; yt=test[TARGET]
pred=pt>=thr
base=average_precision_score(yt,np.full(len(yt),train[TARGET].mean()))
res=pd.Series({'model':best,'threshold':thr,'test_AP':average_precision_score(yt,pt),'baseline_AP(prevalence)':base,
 'test_AUC':roc_auc_score(yt,pt),'precision':precision_score(yt,pred,zero_division=0),'recall':recall_score(yt,pred),'F1':f1_score(yt,pred),'n_test':len(yt),'test_prevalence':yt.mean()})
res

## Conclusion
See the table above for the headline test score (scored once). Interpret test AP against the prevalence baseline, not against 1. `final_notice_sent` would give a very high score but is recorded after the prediction point, so it was excluded deliberately. Limitations: single split of 4 months and a few hundred held-out accounts, so the test estimate has meaningful uncertainty; no calibration or drift analysis was done.